# sech² MCMC — z₀ per (annulus, segment), then polar heatmap

This notebook follows the exact recipe from `Emma-gmm-other-models-demo` and applies it to the real (annulus, segment) data from `setup_state.ipynb`.

For every (annulus, segment) pair we:

1. select stars in that annulus and segment
2. bin in z (weighted-mean + intrinsic-scatter formula, identical to the demo)
3. run NUTS on the mixture-of-sech² model with the demo's exact priors
4. record `z0` median and stddev

Then we assemble the `z0` medians into a polar heatmap, the same way the master notebook plots its quadratic-fit z-minima.

The model, priors, MCMC settings, and binning code are copied verbatim from the demo. The only thing that changes is the data going in.


## 1. Imports + jax config


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binned_statistic

import arviz as az
import numpyro
import numpyro.distributions as dist
import numpyro.infer as infer

jax.config.update("jax_enable_x64", True)
%matplotlib inline


## 2. Rebuild state (same as `setup_state.ipynb`)

If you've just run `setup_state.ipynb` in this kernel and the variables are still in scope, you can skip the next two cells. Otherwise run them — they reproduce the same `t_final`, `star_pack`, `segm`, `crowd_bounds`, error arrays, and binning settings.


In [ ]:
import astropy.table as at
from astropy.table import Table

from src.selection import build_prechem_selection
from src.coords import build_galactocentric_xyz, build_galactocentric_phase_space, build_star_pack, build_segment_setup
from src.chemistry import apply_low_alpha_cut, apply_alfe_mgmn_polygon_cut, FEH_CUT
from src.utils import col_to_float_array

t = Table.read("astraAllStarASPCAP-0.8.fits.gz", format="fits", hdu=2)
t["sdss4_apogee_id"] = t["sdss4_apogee_id"].filled(-1)
dr17 = Table.read("allStarLite-dr17-synspec_rev1.fits", hdu=1)
t = at.join(
    t,
    dr17["APOGEE_ID", "SNR", "VHELIO_AVG"],
    keys_left="sdss4_apogee_id",
    keys_right="APOGEE_ID",
    join_type="left",
)

master_mask, sel_diag, sel_arrays = build_prechem_selection(t)
t_sel = t[master_mask]

x, y, z = build_galactocentric_xyz(t_sel)
chem_result = apply_low_alpha_cut(t_sel, x=x, y=y, z=z, feh_cut=FEH_CUT)
t_sel = chem_result["table"]

t_final, final_diag, final_arrays = apply_alfe_mgmn_polygon_cut(t_sel)

phase, phase_diag = build_galactocentric_phase_space(t_final)

star_pack, star_diag = build_star_pack(t_final, rmax_keep=16, plx_min=None)
R_star    = star_pack["R_star"]
Z_star    = star_pack["Z_star"]
phi_star  = star_pack["phi_star"]
mgfe_star = star_pack["mgfe_star"]
t_star    = star_pack["t_star"]

segm, crowd_bounds, window_edges, seg_diag = build_segment_setup(
    phi_star, phi0=np.pi, crowd_halfwidth_deg=45, n_crowd=11
)
SEG_PLOT_heat = list(range(1, seg_diag["n_crowd"] + 1))

e_mg_h_star = col_to_float_array(t_star, "e_mg_h")
e_fe_h_star = col_to_float_array(t_star, "e_fe_h")

z_edges = np.linspace(-3.0, 3.0, 181)
R_edges = np.arange(0.5, 15.5 + 0.5, 0.5)
minN_seg_total   = 15
min_bin_count    = 5
min_bins_for_fit = 8
z_fit_min, z_fit_max = -1.5, 1.5
zlim = 0.6

print("t_final:", len(t_final), " | star_pack final:", star_diag["final_size"], " | segments:", seg_diag["n_crowd"])


In [ ]:
# per-star [Mg/Fe] error proxy: σ([Mg/Fe]) = sqrt(σ([Mg/H])² + σ([Fe/H])²)
mgfe_err_star = np.sqrt(np.asarray(e_mg_h_star, float)**2 + np.asarray(e_fe_h_star, float)**2)

# bookkeeping for finite values
finite_all = (
    np.isfinite(R_star) & np.isfinite(Z_star) & np.isfinite(phi_star) &
    np.isfinite(mgfe_star) & np.isfinite(mgfe_err_star) & (mgfe_err_star > 0)
)
print("stars with finite (R, Z, phi, mgfe, mgfe_err):", int(finite_all.sum()), "/", len(R_star))


## 3. Model + priors (verbatim from demo)


In [ ]:
def mixture_of_sech2(z, A1, A2, h1, h2, z0):
    # z0 shifts the dip's center along z; the minimum of the curve sits at z = z0
    dz = z - z0
    return A1 * (1 / jnp.cosh(dz / h1)) ** 2 + A2 * (1 / jnp.cosh(dz / h2)) ** 2


In [ ]:
def model(data):
    A1 = numpyro.sample("A1", dist.TruncatedNormal(-1.0, 5.0, low=None, high=0.0))
    A2 = numpyro.sample("A2", dist.TruncatedNormal(-1.0, 5.0, low=None, high=0.0))
    h1 = numpyro.sample("h1", dist.Uniform(0.1, 1.0))
    d_h2 = numpyro.sample("d_h2", dist.Uniform(0.1, 20.0))
    h2 = h1 + d_h2
    z0 = numpyro.sample("z0", dist.Normal(0.0, 1.0))
    offset = numpyro.sample("offset", dist.Normal(0.0, 5.0))

    model_y = mixture_of_sech2(data["z"], A1, A2, h1, h2, z0) + offset
    numpyro.sample("obs", dist.Normal(model_y, data["mg_fe_err"]), obs=data["mg_fe"])


## 4. Per-bin binning + per-segment fit helpers

The binning is the demo's exact code (weighted mean + intrinsic-scatter inflation), wrapped into a function. The MCMC runner uses the demo's exact NUTS settings: `dense_mass=True`, `num_warmup=1000`, `num_samples=1000`, `num_chains=2`.


In [ ]:
# z-bins for the binned-mean fit (same grid the demo uses)
Z_BINS_FIT = np.arange(-5, 5 + 1e-3, 0.1)


def bin_one(z, mgfe, mgfe_err, z_bins=Z_BINS_FIT):
    """Identical binning recipe to the demo."""
    stat1 = binned_statistic(z, mgfe / mgfe_err**2, bins=z_bins, statistic="sum")
    stat2 = binned_statistic(z, 1 / mgfe_err**2, bins=z_bins, statistic="sum")
    stat_std = binned_statistic(z, mgfe, bins=z_bins, statistic="std")
    stat_count = binned_statistic(z, mgfe, bins=z_bins, statistic="count")

    intrinsic_scatter = stat_std.statistic / np.sqrt(stat_count.statistic)

    mean_mg_fe = stat1.statistic / stat2.statistic
    mean_mg_fe_err = np.sqrt(1 / stat2.statistic + intrinsic_scatter**2)
    z_binc = 0.5 * (z_bins[1:] + z_bins[:-1])

    counts = stat_count.statistic
    return z_binc, mean_mg_fe, mean_mg_fe_err, counts


def fit_sech2(z_binc, mean_mg_fe, mean_mg_fe_err, key,
              num_warmup=1000, num_samples=1000, num_chains=2,
              progress_bar=False):
    """Same NUTS settings as the demo. progress_bar off for the loop."""
    sampler = infer.MCMC(
        infer.NUTS(model, dense_mass=True),
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
        progress_bar=progress_bar,
    )
    data = {
        "z": jnp.asarray(z_binc),
        "mg_fe": jnp.asarray(mean_mg_fe),
        "mg_fe_err": jnp.asarray(mean_mg_fe_err),
    }
    sampler.run(key, data=data)
    return sampler


def select_annulus_segment(R_use, Z_use, mgfe_use, mgfe_err_use, seg_mask,
                           Rmin, Rmax,
                           min_bin_count=min_bin_count,
                           min_bins_for_fit=min_bins_for_fit,
                           z_fit_min=z_fit_min, z_fit_max=z_fit_max):
    """Select stars, run the binning, return the data ready to be fed to MCMC.

    Returns (z_binc, mean_mg_fe, mean_mg_fe_err, n_stars, ok) where `ok` is
    False if the (annulus, segment) doesn't have enough data to fit.
    """
    m = (
        seg_mask
        & (R_use >= Rmin) & (R_use < Rmax)
        & np.isfinite(Z_use) & np.isfinite(mgfe_use)
        & np.isfinite(mgfe_err_use) & (mgfe_err_use > 0)
    )
    n_stars = int(m.sum())
    if n_stars < minN_seg_total:
        return None, None, None, n_stars, False

    z_binc, mean_mg_fe, mean_mg_fe_err, counts = bin_one(
        Z_use[m], mgfe_use[m], mgfe_err_use[m]
    )

    # demo-style: drop bins with NaN/Inf or too few stars; restrict to fit window
    good = (
        np.isfinite(mean_mg_fe) & np.isfinite(mean_mg_fe_err)
        & (counts >= min_bin_count)
        & (z_binc >= z_fit_min) & (z_binc <= z_fit_max)
    )
    if good.sum() < min_bins_for_fit:
        return None, None, None, n_stars, False

    return (
        np.asarray(z_binc[good], float),
        np.asarray(mean_mg_fe[good], float),
        np.asarray(mean_mg_fe_err[good], float),
        n_stars,
        True,
    )


## 5. Smoke test on one (annulus, segment)

Pick one (annulus, segment) pair and run through the full demo workflow: bin, fit, posterior summary, posterior-predictive plot, `z0` median/stddev. This validates the pipeline before the full loop.


In [ ]:
# pick a well-populated annulus and a crowded segment
Rmin_focus, Rmax_focus = 8.0, 8.5
seg_focus = 6  # any of 1..seg_diag['n_crowd']

z_binc_t, mean_mg_fe_t, mean_mg_fe_err_t, n_stars_t, ok_t = select_annulus_segment(
    R_star, Z_star, mgfe_star, mgfe_err_star,
    seg_mask=segm[seg_focus],
    Rmin=Rmin_focus, Rmax=Rmax_focus,
)
print(f"stars in segment {seg_focus}, R=[{Rmin_focus},{Rmax_focus}]:", n_stars_t, "| fittable bins:", len(z_binc_t) if ok_t else 0)

assert ok_t, "smoke-test bin too sparse — pick a different (annulus, segment)"

sampler_t = fit_sech2(z_binc_t, mean_mg_fe_t, mean_mg_fe_err_t, jr.PRNGKey(0), progress_bar=True)
samples_t = sampler_t.get_samples()
idata_t   = az.from_numpyro(sampler_t)
az.summary(idata_t)


In [ ]:
# posterior predictive plot — same look as demo cell 15
z_grid = np.linspace(-5, 5, 1000)

plt.figure(figsize=(6, 4))
plt.errorbar(
    z_binc_t, mean_mg_fe_t, yerr=mean_mg_fe_err_t,
    color="C2", linestyle="", marker=".",
)
for i in range(100):
    A1 = samples_t["A1"][i]
    A2 = samples_t["A2"][i]
    h1 = samples_t["h1"][i]
    h2 = h1 + samples_t["d_h2"][i]
    z0 = samples_t["z0"][i]
    y_pred = mixture_of_sech2(z_grid, A1, A2, h1, h2, z0) + samples_t["offset"][i]
    plt.plot(z_grid, y_pred, color="C1", alpha=0.1, marker="")

plt.xlabel("z")
plt.ylabel("[Mg/Fe]")
plt.title(f"segment {seg_focus}, R=[{Rmin_focus},{Rmax_focus}] kpc")
plt.show()

print(f"z0 median = {np.median(samples_t['z0']):+.4f}")
print(f"z0 stddev = {np.std(samples_t['z0']):.3f}")
SNR_z0 = np.abs(np.median(samples_t['z0']) / np.std(samples_t['z0']))
print(f"SNR(z0)   = {SNR_z0:.2f}")


## 6. Full loop over all (annulus, segment) pairs

Runs the demo's MCMC for every (annulus, segment) pair and stores `z0` median + stddev plus the star count per cell.

Heads up: this is many fits. After the first fit JIT-compiles (~30 s), subsequent fits are a few seconds each. The crowded-segments-only loop is `n_R × n_seg = (len(R_edges)-1) × seg_diag['n_crowd']`. Plan accordingly; you can lower `num_samples` / `num_warmup` here if you want a quick run, but I'm leaving the demo settings unchanged.


In [ ]:
from itertools import product

n_R   = len(R_edges) - 1
n_seg = len(SEG_PLOT_heat)

z0_med   = np.full((n_R, n_seg), np.nan)
z0_std   = np.full((n_R, n_seg), np.nan)
n_stars  = np.zeros((n_R, n_seg), dtype=int)
n_bins   = np.zeros((n_R, n_seg), dtype=int)
ok_grid  = np.zeros((n_R, n_seg), dtype=bool)

key = jr.PRNGKey(0)

pairs = list(product(range(n_R), range(n_seg)))
print(f"running {len(pairs)} (annulus, segment) fits ...")

for k, (i_R, j_seg) in enumerate(pairs, 1):
    Rmin = float(R_edges[i_R])
    Rmax = float(R_edges[i_R + 1])
    seg_id = SEG_PLOT_heat[j_seg]

    z_binc_k, mean_k, err_k, ns_k, ok_k = select_annulus_segment(
        R_star, Z_star, mgfe_star, mgfe_err_star,
        seg_mask=segm[seg_id],
        Rmin=Rmin, Rmax=Rmax,
    )
    n_stars[i_R, j_seg] = ns_k
    if not ok_k:
        continue

    n_bins[i_R, j_seg] = len(z_binc_k)
    try:
        key, sub = jr.split(key)
        sampler_k = fit_sech2(z_binc_k, mean_k, err_k, sub, progress_bar=False)
        samples_k = sampler_k.get_samples()
        z0_med[i_R, j_seg] = float(np.median(samples_k["z0"]))
        z0_std[i_R, j_seg] = float(np.std(samples_k["z0"]))
        ok_grid[i_R, j_seg] = True
    except Exception as e:
        print(f"  fit failed: seg={seg_id} R=[{Rmin:.1f},{Rmax:.1f}] -> {e}")

    if k % 10 == 0 or k == len(pairs):
        print(f"  done {k}/{len(pairs)}")

print("fits complete:", int(ok_grid.sum()), "/", ok_grid.size)


In [ ]:
# save results so the loop doesn't have to be rerun
import os
out_npz = "sech2_mcmc_z0_results.npz"
np.savez(
    out_npz,
    R_edges=R_edges,
    SEG_PLOT_heat=np.asarray(SEG_PLOT_heat),
    crowd_bounds=np.asarray(crowd_bounds),
    z0_med=z0_med,
    z0_std=z0_std,
    n_stars=n_stars,
    n_bins=n_bins,
    ok_grid=ok_grid,
)
print("saved:", os.path.abspath(out_npz))


## 7. Polar heatmap (matches the original notebook's style)

Build `theta_edges` from `crowd_bounds` (unwrapped, same logic as cell 16 of the master notebook), then plot a polar `pcolormesh` of `z0_med` over (R, θ). Cells with no fit appear as masked / nan.


In [ ]:
# build unwrapped theta edges from crowd_bounds (same construction as the master notebook)
theta_edges = np.unwrap(np.array(crowd_bounds, float))
for i in range(1, len(theta_edges)):
    while theta_edges[i] <= theta_edges[i - 1]:
        theta_edges[i] += 2 * np.pi

assert len(theta_edges) - 1 == n_seg, "theta_edges length must match number of crowded segments"

# meshgrid for polar pcolormesh
T_mesh, R_mesh = np.meshgrid(theta_edges, R_edges)

Zmap_med = np.ma.masked_invalid(z0_med)
Zmap_std = np.ma.masked_invalid(z0_std)

fig = plt.figure(figsize=(15, 7))

# left: z0 median
ax1 = plt.subplot(1, 2, 1, projection="polar")
pcm1 = ax1.pcolormesh(T_mesh, R_mesh, Zmap_med, cmap="RdBu_r", vmin=-zlim, vmax=zlim, shading="flat")
fig.colorbar(pcm1, ax=ax1, pad=0.1, label=r"$z_0$ median [kpc]")
ax1.set_title(r"sech$^2$ MCMC: $z_0$ (median) per (annulus, segment)")
ax1.set_theta_zero_location("E")
ax1.set_theta_direction(1)
ax1.set_rmax(R_edges[-1])

# right: z0 stddev
ax2 = plt.subplot(1, 2, 2, projection="polar")
pcm2 = ax2.pcolormesh(T_mesh, R_mesh, Zmap_std, cmap="viridis", vmin=0.0, vmax=np.nanpercentile(z0_std, 95) if np.isfinite(z0_std).any() else 0.5, shading="flat")
fig.colorbar(pcm2, ax=ax2, pad=0.1, label=r"$z_0$ stddev [kpc]")
ax2.set_title(r"sech$^2$ MCMC: $z_0$ (stddev) per (annulus, segment)")
ax2.set_theta_zero_location("E")
ax2.set_theta_direction(1)
ax2.set_rmax(R_edges[-1])

plt.tight_layout()
plt.show()


## 8. (optional) reload + re-plot

If you re-open the notebook later and just want to remake the heatmap from saved results:


In [ ]:
# d = np.load("sech2_mcmc_z0_results.npz")
# z0_med = d["z0_med"]; z0_std = d["z0_std"]
# R_edges = d["R_edges"]; crowd_bounds = d["crowd_bounds"]
# SEG_PLOT_heat = list(d["SEG_PLOT_heat"])
# # then re-run the polar heatmap cell above
